# Train và so sánh tồn kho M5 — mở notebook này và chọn **Run All**

Ba phương pháp: **(R,S)**, **(R,s,S)** và **ASL-LS (LightGBM)**. Mỗi SKU có S/s hoặc model riêng; cuối notebook có bảng tổng hợp.

**Cách chạy trong VS Code:**
1. Chọn **Select Kernel → Python Environments → `.venv\Scripts\python.exe`** của dự án.
2. Sửa ô **CẤU HÌNH** bên dưới nếu cần.
3. Chọn **Run All**. Đọc bảng dữ liệu ở bước 3 và bảng kết quả ở bước 9.

Notebook này tự chứa toàn bộ code, không cần thư mục `baseline/`, YAML hay module riêng.
Nếu máy mới chưa có thư viện, chạy trong một ô notebook: `%pip install numpy pandas lightgbm==4.6.0` rồi khởi động lại kernel.

**File sử dụng:** chỉ đọc ba CSV M5 sẵn có trong `data/m5-forecasting-accuracy/` ở gốc dự án. `sample_submission.csv` không dùng để train. Mặc định không tạo file phụ; lưu notebook sau khi chạy sẽ giữ các bảng output.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Sequence
from time import perf_counter
import csv
import hashlib
import math
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display

# Tìm gốc dự án dù mở notebook từ VS Code hay Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/m5-forecasting-accuracy").is_dir())
DATA_DIR = ROOT / "data/m5-forecasting-accuracy"
NOTEBOOK_DIR = ROOT / "notebooks/baslines"
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 25)
print("Python:", sys.version.split()[0], "| LightGBM:", lgb.__version__)
print("Đọc dữ liệu từ:", DATA_DIR)

Python: 3.11.9 | LightGBM: 4.6.0
Đọc dữ liệu từ: D:\PY\Inventory-Management\data\m5-forecasting-accuracy


## 1. CẤU HÌNH — chỉ cần sửa ô này rồi Run All

`STORE_ID` là cửa hàng. `N_SKUS=None` nghĩa là chạy toàn bộ SKU tại cửa hàng đó có đủ lịch sử train hợp lệ; đặt một số nguyên (ví dụ 20) chỉ để chạy thử nhanh. Mặc định chạy toàn bộ CA_1, chưa phải toàn bộ M5.

`L` là số ngày chờ hàng, `R` là số ngày giữa hai lần xét đặt. ASL-LS ở đây yêu cầu **R=1**.
`H` là phí giữ một đơn vị trong một ngày; `P` là phạt một đơn vị nhu cầu mất; `K_VALUES` là phí cố định cho một đơn đặt có lượng >0. Đây là **giả định thí nghiệm**, không lấy từ M5.

In [ ]:
STORE_ID = "CA_1"
N_SKUS = None  # None: toàn bộ SKU đủ lịch sử train tại CA_1; 20: chạy thử nhanh.
SEED = 2024
SALES_FILE = "sales_train_evaluation.csv"  # Có thể đổi sang sales_train_validation.csv.

VAL_DAYS = 56
TEST_DAYS = 112
WARMUP_DAYS = 28   # Chạy trước để ổn định tồn kho; không tính vào kết quả.
MIN_TRAIN_DAYS = 365

L = 2             # Đặt ngày t -> nhận đầu ngày t+2.
R_CANDIDATES = [1, 2, 3, 7]  # Chu kỳ xét đặt thử cho (R,S) và (R,s,S).
ASL_R = 1         # ASL-LS gốc chỉ hợp lệ với quyết định hằng ngày.
INITIAL_STOCK = 0
INITIAL_PIPELINE = {}  # Hàng đang đi lúc bắt đầu: {ngày_nhận: số_lượng}.
H = 0.1
P = 5.0
K_VALUES = [0.0, 5.0]
PURCHASE_RATIO = 0.7  # Giá mua giả định = 70% trung vị giá bán trên TRAIN.

# Ngân sách tìm kiếm/huấn luyện; có thể giữ nguyên khi mới chạy.
GRID_SIZE = 12
MAX_EXPANSIONS = 3
MIN_FILL_RATE = None  # Ví dụ 0.95 nếu yêu cầu fill rate validation >=95%.
ASL_ROUNDS = 160
ASL_AUGMENT = 3
ASL_INVENTORY_BOUNDS = [2.0, 4.0]

SAVE_RESULTS = False  # True: lưu đúng MỘT file ket_qua.pkl.gz, không xuất data đã xử lý.

assert ASL_R == 1, "ASL-LS gốc trong notebook này chỉ dùng ASL_R=1."
assert R_CANDIDATES and all(isinstance(r, int) and r >= 1 for r in R_CANDIDATES)
assert len(set(R_CANDIDATES)) == len(R_CANDIDATES)
assert isinstance(L, int) and L >= 1 and (N_SKUS is None or (isinstance(N_SKUS, int) and N_SKUS > 0))
assert 0 <= WARMUP_DAYS <= VAL_DAYS and TEST_DAYS > 0 and VAL_DAYS > 0
assert H > 0 and P >= 0 and 0 in K_VALUES and all(k >= 0 for k in K_VALUES)
assert len(set(K_VALUES)) == len(K_VALUES) and 0 < PURCHASE_RATIO <= 1
assert MIN_FILL_RATE is None or 0 <= MIN_FILL_RATE <= 1
assert GRID_SIZE >= 3 and MAX_EXPANSIONS >= 0 and ASL_ROUNDS >= 1 and ASL_AUGMENT >= 1
assert ASL_INVENTORY_BOUNDS and all(v > 0 for v in ASL_INVENTORY_BOUNDS)
assert INITIAL_STOCK >= 0 and all(isinstance(d, int) and 0 <= d <= L and q >= 0
                                for d, q in INITIAL_PIPELINE.items())
CONFIG = {name: globals()[name] for name in [
    "STORE_ID", "N_SKUS", "SEED", "SALES_FILE", "VAL_DAYS", "TEST_DAYS", "WARMUP_DAYS",
    "MIN_TRAIN_DAYS", "L", "R_CANDIDATES", "ASL_R", "INITIAL_STOCK", "INITIAL_PIPELINE", "H", "P", "K_VALUES",
    "PURCHASE_RATIO", "GRID_SIZE", "MAX_EXPANSIONS", "MIN_FILL_RATE", "ASL_ROUNDS",
    "ASL_AUGMENT", "ASL_INVENTORY_BOUNDS", "SAVE_RESULTS"]}
display(pd.Series(CONFIG, name="Giá trị").to_frame())

,Giá trị
STORE_ID,CA_1
N_SKUS,20
SEED,2024
SALES_FILE,sales_train_evaluation.csv
VAL_DAYS,56
TEST_DAYS,112
WARMUP_DAYS,28
MIN_TRAIN_DAYS,365
L,2
R,1


## 2. Dữ liệu được xử lý như thế nào?

**Không sửa giá trị trong CSV gốc, không tạo bảng data mới trên ổ đĩa.** Tất cả các bước sau chỉ ở RAM:

1. Đọc sales theo chunk, kiểm tra khóa trùng/thiếu và sales không âm. Giữ một cửa hàng, chọn SKU dựa vào **train** và seed.
2. Nối cột `d_1`, `d_2`… với `calendar.d` để có ngày, thứ, tháng, sự kiện và SNAP.
3. Nối giá bán theo **store_id + item_id + wm_yr_wk**. Không dùng giá bán làm giá mua.
4. Giữ NaN giá trước khi SKU lên kệ; không điền bằng giá tương lai. Thiếu giá sau khi lên kệ thì báo lỗi. Khi train, bỏ khoảng trước ngày đầu có giá của từng SKU.
5. Chia thời gian theo các cột sales thực có: phần đầu là train, tiếp theo 56 ngày validation, cuối cùng 112 ngày test. Không dùng test để chọn tham số.

**Giả định quan trọng:** M5 ghi sales quan sát. Ở đây coi sales đó là nhu cầu ngoại sinh cho mô phỏng, không khẳng định đã tìm lại nhu cầu thật khi Walmart hết hàng.

Ô hàm dưới đây chỉ phục vụ đọc/kiểm tra CSV; không cần sửa khi chạy thông thường.

In [ ]:
KEYS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

@dataclass
class M5Dataset:
    demand: np.ndarray
    prices: np.ndarray
    calendar: pd.DataFrame
    items: pd.DataFrame
    audit: dict


def _header(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return next(csv.reader(handle))


def _calendar(path: Path, day_columns: list[str], state_id: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {"d", "date", "wm_yr_wk", "wday", "month", f"snap_{state_id}"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Calendar missing columns: {required - set(frame.columns)}")
    if frame["d"].isna().any() or frame["d"].duplicated().any():
        raise ValueError("Calendar d must be unique and nonmissing")
    if frame["date"].isna().any() or frame["date"].duplicated().any():
        raise ValueError("Calendar dates must be unique and nonmissing")
    indexed = frame.set_index("d")
    missing = set(day_columns) - set(indexed.index)
    if missing:
        raise ValueError(f"Calendar missing sales days: {sorted(missing)[:5]}")
    selected = indexed.loc[day_columns].reset_index()
    selected["date"] = pd.to_datetime(selected["date"], errors="raise")
    if not selected["date"].diff().iloc[1:].eq(pd.Timedelta(days=1)).all():
        raise ValueError("Selected sales calendar is not consecutive daily data")
    selected["weekday"] = selected["wday"].astype("int8")
    selected["event"] = selected.get("event_name_1", pd.Series(index=selected.index, dtype=object)).notna().astype("int8")
    selected["snap"] = selected[f"snap_{state_id}"].astype("int8")
    if selected[["wm_yr_wk", "weekday", "month", "snap"]].isna().any().any():
        raise ValueError("Calendar join keys and known features must be nonmissing")
    return selected


def load_m5(
    data_dir: str | Path,
    store_id: str = "CA_1",
    n_skus: int | None = None,
    seed: int = 2024,
    validation_days: int = 56,
    test_days: int = 112,
    min_train_days: int = 365,
    item_ids: Sequence[str] | None = None,
    sales_file: str | None = None,
) -> M5Dataset:
    """Load one store and a seeded subset, with an audit of actual file bounds.

    Sampling eligibility uses only train observations: first positive sale must
    leave at least ``min_train_days`` before the train boundary. It never uses
    validation/test sales to select SKUs. Explicit item_ids override sampling.
    CSV sales are scanned in chunks twice; only selected rows are retained.
    Price duplicate/missing checks cover selected store/item pairs; sales key
    and demand checks cover the whole source table.
    """
    root = Path(data_dir)
    if sales_file is None:
        sales_file = next((name for name in ("sales_train_evaluation.csv", "sales_train_validation.csv") if (root / name).exists()), None)
    if sales_file is None:
        raise FileNotFoundError("No M5 sales_train_evaluation/validation.csv found")
    sales_path = root / sales_file
    columns = _header(sales_path)
    if not set(KEYS).issubset(columns):
        raise ValueError(f"Sales table missing keys: {set(KEYS) - set(columns)}")
    days = sorted((name for name in columns if name.startswith("d_") and name[2:].isdigit()), key=lambda name: int(name[2:]))
    if not days or [int(name[2:]) for name in days] != list(range(1, len(days) + 1)):
        raise ValueError("Sales d_* columns must be consecutive from d_1")
    n_days = len(days)
    train_end = n_days - int(validation_days) - int(test_days)
    if min(validation_days, test_days, min_train_days) < 1 or train_end < min_train_days:
        raise ValueError("Not enough actual labeled days for configured chronological split")
    if n_skus is not None and n_skus < 1:
        raise ValueError("n_skus must be positive or None for all eligible SKUs")
    dtype = {name: "string" for name in KEYS}
    dtype.update({name: "float32" for name in days})
    seen: set[tuple[str, str]] = set()
    seen_ids: set[str] = set()
    store_items: dict[str, set[str]] = {}
    candidates: list[dict] = []
    row_count = 0
    for chunk in pd.read_csv(sales_path, usecols=KEYS + days, dtype=dtype, chunksize=512):
        if chunk[KEYS].isna().any().any():
            raise ValueError("Missing sales identifier")
        values = chunk[days].to_numpy(dtype=np.float32)
        if not np.isfinite(values).all() or (values < 0).any() or (values != np.floor(values)).any():
            raise ValueError("M5 simulated demand must be finite nonnegative integer observations")
        for row in chunk[KEYS].itertuples(index=False, name=None):
            identifier, item, _, _, store, _ = row
            if (store, item) in seen or identifier in seen_ids:
                raise ValueError(f"Duplicate sales id or store/item: {store}/{item}")
            seen.add((store, item))
            seen_ids.add(identifier)
            store_items.setdefault(store, set()).add(item)
        row_count += len(chunk)
        selected = chunk.loc[chunk["store_id"].eq(store_id)]
        for _, row in selected.iterrows():
            observed = row[days].to_numpy(dtype=np.float32)
            positive = np.flatnonzero(observed > 0)
            first_positive = int(positive[0] + 1) if len(positive) else None
            metadata = {key: str(row[key]) for key in KEYS}
            metadata["first_positive_day"] = first_positive
            metadata["eligible_from_train"] = first_positive is not None and first_positive <= train_end - min_train_days + 1
            candidates.append(metadata)
    if not candidates:
        raise ValueError(f"Store {store_id!r} absent; available stores: {sorted(store_items)}")
    by_item = {entry["item_id"]: entry for entry in candidates}
    eligible = sorted(entry["item_id"] for entry in candidates if entry["eligible_from_train"])
    if item_ids is None:
        if n_skus is None:
            chosen = eligible
        else:
            if n_skus > len(eligible):
                raise ValueError(f"Requested {n_skus} SKUs but only {len(eligible)} eligible in train")
            chosen = sorted(np.random.default_rng(seed).choice(eligible, size=n_skus, replace=False).tolist())
    else:
        chosen = list(item_ids)
        if len(set(chosen)) != len(chosen) or not chosen:
            raise ValueError("Explicit item_ids must be nonempty and unique")
        missing_items = set(chosen) - set(by_item)
        if missing_items:
            raise ValueError(f"Unknown items in selected store: {sorted(missing_items)}")
        inactive = [item for item in chosen if not by_item[item]["eligible_from_train"]]
        if inactive:
            raise ValueError(f"Explicit items lack configured training history: {inactive}")
    rows = []
    for chunk in pd.read_csv(sales_path, usecols=KEYS + days, dtype=dtype, chunksize=512):
        subset = chunk.loc[chunk["store_id"].eq(store_id) & chunk["item_id"].isin(chosen)]
        if len(subset):
            rows.append(subset)
    selected_sales = pd.concat(rows).set_index("item_id").loc[chosen]
    demand = selected_sales[days].to_numpy(dtype=np.float32)
    items = pd.DataFrame([by_item[item] for item in chosen])
    if items["state_id"].nunique() != 1:
        raise ValueError("One store must map to exactly one state")
    calendar = _calendar(root / "calendar.csv", days, items["state_id"].iloc[0])
    price_rows = []
    price_count = 0
    price_columns = ["store_id", "item_id", "wm_yr_wk", "sell_price"]
    for chunk in pd.read_csv(root / "sell_prices.csv", usecols=price_columns, dtype={"store_id": "string", "item_id": "string", "wm_yr_wk": "int32", "sell_price": "float32"}, chunksize=250000):
        price_count += len(chunk)
        subset = chunk.loc[chunk["store_id"].eq(store_id) & chunk["item_id"].isin(chosen)]
        if len(subset):
            price_rows.append(subset)
    if not price_rows:
        raise ValueError("No retail price observations for selected items")
    prices_table = pd.concat(price_rows, ignore_index=True)
    if prices_table[price_columns].isna().any().any():
        raise ValueError("Missing selected price key/value")
    if prices_table.duplicated(["store_id", "item_id", "wm_yr_wk"]).any():
        raise ValueError("Duplicate selected store/item/week retail prices")
    if not np.isfinite(prices_table["sell_price"]).all() or (prices_table["sell_price"] <= 0).any():
        raise ValueError("Retail prices must be finite and positive")
    prices = np.full_like(demand, np.nan)
    price_audit = []
    for index, item in enumerate(chosen):
        item_prices = prices_table.loc[prices_table["item_id"].eq(item)].set_index("wm_yr_wk")["sell_price"]
        prices[index] = calendar["wm_yr_wk"].map(item_prices).to_numpy(dtype=np.float32)
        available = np.flatnonzero(np.isfinite(prices[index]))
        first_priced = int(available[0] + 1) if len(available) else None
        first_priced_week = int(item_prices.index.min())
        if ((demand[index] > 0) & ~np.isfinite(prices[index])).any():
            raise ValueError(f"Positive sales without observed same-week retail price: {item}")
        after_launch_missing = int(np.isnan(prices[index, first_priced - 1:]).sum()) if first_priced else n_days
        if after_launch_missing:
            raise ValueError(f"Missing retail prices after launch for {item}: {after_launch_missing} days; choose complete items or explicitly configure a causal missing-price policy")
        items.loc[index, "first_priced_day"] = first_priced
        items.loc[index, "first_priced_index"] = first_priced - 1 if first_priced else None
        items.loc[index, "first_priced_week"] = first_priced_week
        price_audit.append({"item_id": item, "first_positive_day": by_item[item]["first_positive_day"], "first_priced_day": first_priced, "first_priced_week": first_priced_week, "missing_price_days": int(np.isnan(prices[index]).sum()), "missing_price_days_after_launch": after_launch_missing})
    def split(start: int, stop: int) -> dict:
        return {"first_day": days[start], "last_day": days[stop - 1], "n_days": stop - start, "first_date": calendar["date"].iloc[start].date().isoformat(), "last_date": calendar["date"].iloc[stop - 1].date().isoformat()}
    audit = {
        "sales_file": str(sales_path),
        "source_sales_rows": row_count,
        "stores": {store: len(item_set) for store, item_set in sorted(store_items.items())},
        "sales_days": n_days,
        "first_day": days[0], "last_day": days[-1],
        "first_date": calendar["date"].iloc[0].date().isoformat(), "last_date": calendar["date"].iloc[-1].date().isoformat(),
        "selected_store": store_id, "selected_skus": chosen, "selected_sku_count": len(chosen),
        "selection_seed": seed, "eligible_sku_count_from_train": len(eligible), "min_train_days": min_train_days,
        "selection_rule": ("all items whose first positive observed sale leaves min_train_days before train boundary; no validation/test selection" if n_skus is None else "seeded sample of eligible items using train only; no validation/test selection"),
        "splits": {"train": split(0, train_end), "validation": split(train_end, n_days - test_days), "test": split(n_days - test_days, n_days)},
        "sales_key_duplicates": 0, "sales_missing_keys": 0, "sales_missing_or_invalid_demand": 0,
        "calendar_d_duplicates": 0, "calendar_missing_sales_days": 0,
        "price_source_rows": price_count, "selected_price_rows": len(prices_table),
        "price_checks_scope": "selected store/item pairs; all source rows streamed, no global duplicate-key set allocated",
        "selected_price_key_duplicates": 0, "selected_price_missing_keys": 0,
        "price_join": ["store_id", "item_id", "wm_yr_wk"], "calendar_join": "d_* -> calendar.d",
        "launch_and_price_missing": price_audit,
        "price_missing_policy": "preserve prelaunch NaN; no forward/future backfill; reject missing prices after launch and positive sales without same-week observed price",
        "demand_assumption": "historical observed M5 sales treated as exogenous simulation demand, not recovered uncensored demand",
    }
    return M5Dataset(demand, prices, calendar, items, audit)

## 3. Đọc CSV và xem đúng dữ liệu đang dùng

Giá mua chỉ là giả định để tính GMROI; chi phí mục tiêu chỉ gồm giữ hàng + phạt thiếu + phí đặt.
Giá bán, calendar và SNAP hiện tại được giả định đã biết trước khi đặt hàng.

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_files = [DATA_DIR / SALES_FILE, DATA_DIR / "calendar.csv", DATA_DIR / "sell_prices.csv"]
original_hashes = {path.name: file_hash(path) for path in source_files}
dataset = load_m5(DATA_DIR, store_id=STORE_ID, n_skus=N_SKUS, seed=SEED,
                  validation_days=VAL_DAYS, test_days=TEST_DAYS,
                  min_train_days=MIN_TRAIN_DAYS, sales_file=SALES_FILE)
items, demand_matrix, price_matrix, calendar = (
    dataset.items, dataset.demand, dataset.prices, dataset.calendar)
N_DAYS = demand_matrix.shape[1]
TRAIN_END = N_DAYS - VAL_DAYS - TEST_DAYS
TEST_START = N_DAYS - TEST_DAYS
assert TRAIN_END - WARMUP_DAYS >= 28
splits = pd.DataFrame([
    {"Tập": name, "Từ": calendar.date.iloc[start].date(),
     "Đến": calendar.date.iloc[end - 1].date(), "Số ngày": end - start}
    for name, start, end in [("Train", 0, TRAIN_END), ("Validation", TRAIN_END, TEST_START),
                              ("Test", TEST_START, N_DAYS)]])
print(f"Nguồn: {dataset.audit['source_sales_rows']:,} chuỗi, "
      f"{len(dataset.audit['stores'])} cửa hàng; đang chọn {len(items)} SKU tại {STORE_ID}.")
print("Không có khóa trùng, demand lỗi hoặc thiếu giá sau khi SKU lên kệ trong tập được chọn.")
display(splits)
display(items[["item_id", "store_id", "dept_id", "cat_id", "first_positive_day", "first_priced_day"]])

Nguồn: 30,490 chuỗi, 10 cửa hàng; đang chọn 20 SKU tại CA_1.
Không có khóa trùng, demand lỗi hoặc thiếu giá sau khi SKU lên kệ trong tập được chọn.


,Tập,Từ,Đến,Số ngày
0,Train,2011-01-29,2015-12-06,1773
1,Validation,2015-12-07,2016-01-31,56
2,Test,2016-02-01,2016-05-22,112


,item_id,store_id,dept_id,cat_id,first_positive_day,first_priced_day
0,FOODS_2_019,CA_1,FOODS_2,FOODS,1,1.0
1,FOODS_2_020,CA_1,FOODS_2,FOODS,1,1.0
2,FOODS_2_059,CA_1,FOODS_2,FOODS,1,1.0
3,FOODS_2_216,CA_1,FOODS_2,FOODS,46,43.0
4,FOODS_2_289,CA_1,FOODS_2,FOODS,1,1.0
5,FOODS_2_300,CA_1,FOODS_2,FOODS,427,421.0
6,FOODS_2_333,CA_1,FOODS_2,FOODS,1,1.0
7,FOODS_3_034,CA_1,FOODS_3,FOODS,932,932.0
8,FOODS_3_116,CA_1,FOODS_3,FOODS,1,1.0
9,FOODS_3_191,CA_1,FOODS_3,FOODS,1,1.0


## 4. Simulator chung và hai policy cổ điển

Một ngày chạy theo thứ tự: **nhận hàng → xem tồn/pipeline → đặt hàng → phục vụ nhu cầu → tính tồn cuối ngày và chi phí**.
Đơn đặt hôm nay chỉ tới đầu ngày `t+L`. Nhu cầu không phục vụ bị mất, không ghi nợ sang hôm sau.

- **(R,S):** đặt `max(0, S − IP)` vào ngày review.
- **(R,s,S):** chỉ đặt khi **IP < s**, rồi đặt `max(0, S − IP)`; luôn s < S.
- **IP** = tồn đã nhận được + hàng chưa đến. Policy chỉ thấy sales đã phục vụ của các ngày trước, không thấy demand hôm nay/tương lai.

Tồn đầu và pipeline đầu giống nhau cho mọi policy. Chạy warm-up trước mỗi khoảng chấm điểm rồi bỏ chi phí warm-up. Không tính chi phí mua, không cộng salvage; báo riêng tồn cuối và hàng còn đang đi.

In [ ]:
def classical_policy(S, s=None):
    def policy(state):
        ip = state["stock"] + sum(state["pipeline"])
        return max(0.0, S - ip) if s is None or ip < s else 0.0
    return policy


def simulate(demand, policy, *, known=None, prices=None, unit_cost=1.0, history=(),
             warmup=0, k=0.0, lead_time=L, review=1, h=H, p=P,
             initial_stock=INITIAL_STOCK, initial_pipeline=None):
    demand = np.asarray(demand, dtype=float)
    assert demand.ndim == 1 and len(demand) > warmup >= 0
    assert np.isfinite(demand).all() and (demand >= 0).all()
    assert lead_time >= 1 and review >= 1
    known = [{} for _ in demand] if known is None else known
    prices = np.full(len(demand), np.nan) if prices is None else np.asarray(prices)
    assert len(known) == len(demand) == len(prices)
    pipeline = dict(INITIAL_PIPELINE if initial_pipeline is None else initial_pipeline)
    observed, stock, rows = list(history), float(initial_stock), []
    for t, d in enumerate(demand):
        received = pipeline.pop(t, 0.0)
        stock += received
        outstanding = tuple(pipeline.get(t + j, 0.0) for j in range(1, lead_time + 1))
        state = {"stock": stock, "pipeline": outstanding,
                 "history": tuple(observed), "known": dict(known[t])}
        q, seconds = 0.0, 0.0
        if t % review == 0:
            started = perf_counter()
            proposed = float(policy(state))
            seconds = perf_counter() - started
            assert np.isfinite(proposed) and proposed >= 0
            q = float(np.rint(proposed))
            if q > 0:
                pipeline[t + lead_time] = pipeline.get(t + lead_time, 0.0) + q
        sales = min(stock, d)
        lost = d - sales
        end = stock - sales
        holding, penalty, ordering = h * end, p * lost, k if q > 0 else 0.0
        margin = sales * (prices[t] - unit_cost) if np.isfinite(prices[t]) else (0 if sales == 0 else np.nan)
        rows.append(dict(day=t, review=t % review == 0, cycle=t // review, review_period=review,
                         received=received, on_hand=stock, ip=stock + sum(outstanding),
                         pipeline=outstanding, order=q, arrival_day=t + lead_time if q else np.nan,
                         demand=d, sales=sales, lost=lost, end_inventory=end,
                         on_order=sum(pipeline.values()), pipeline_due=tuple(sorted(pipeline.items())),
                         holding_cost=holding, lost_sales_cost=penalty, ordering_cost=ordering,
                         total_cost=holding + penalty + ordering, inventory_value=end * unit_cost,
                         gross_margin=margin, inference_seconds=seconds, scored=t >= warmup))
        observed.append(sales)  # Policy chỉ quan sát lượng bán đã phục vụ.
        stock = end
    return pd.DataFrame(rows)


def ratio(numerator, denominator):
    return float(numerator / denominator) if denominator > 0 else np.nan


def summarize(log):
    scored = log[log.scored]
    n = len(scored)
    cycles = scored.groupby("cycle").agg(size=("day", "size"), begins=("review", "first"),
                                          lost=("lost", "sum"))
    complete = cycles[(cycles["size"] == int(log.review_period.iloc[0])) & cycles.begins]
    total = lambda col: float(scored[col].sum())
    inventory_value = total("inventory_value") / n
    margin = float(scored.gross_margin.sum(min_count=n))
    result = {name: total(name) for name in ["holding_cost", "lost_sales_cost", "ordering_cost", "total_cost", "inference_seconds"]}
    result.update(days=n, total_demand=total("demand"), total_sales=total("sales"), total_lost=total("lost"),
                  fill_rate=ratio(total("sales"), total("demand")), complete_cycles=len(complete),
                  successful_cycles=int((complete.lost == 0).sum()),
                  csl=ratio(int((complete.lost == 0).sum()), len(complete)),
                  average_end_inventory=total("end_inventory") / n,
                  average_inventory_value=inventory_value, gross_margin=margin,
                  gmroi=ratio(margin, inventory_value), order_count=int((scored.order > 0).sum()),
                  total_ordered=total("order"), terminal_inventory=float(log.end_inventory.iloc[-1]),
                  terminal_pipeline=float(log.on_order.iloc[-1]))
    return result

## 5. Chọn S và (s,S) bằng validation

Miền tìm kiếm ban đầu tính từ tổng sales train trong L+R ngày. Mỗi ứng viên chạy cùng validation và warm-up; chọn chi phí nhỏ nhất (hoặc đạt `MIN_FILL_RATE` trước). Nếu nghiệm chạm biên S cao nhất, mở rộng miền. Đây là tìm kiếm theo grid, không chứng minh tối ưu toàn cục. S/s sau khi chọn được giữ cố định trong test.

In [ ]:
def selection_key(metrics):
    fill = metrics["fill_rate"]
    shortfall = 0.0 if MIN_FILL_RATE is None or not np.isfinite(fill) else max(0, MIN_FILL_RATE - fill)
    return shortfall, metrics["total_cost"]


def tune_classical_fixed_r(train, validation_inputs, k, review):
    train = np.asarray(train, dtype=float)
    protection = min(len(train), L + review)
    c = np.r_[0.0, np.cumsum(train)]
    upper = max(2, int(np.ceil(1.5 * np.quantile(c[protection:] - c[:-protection], 0.99))))
    records, seen = [], set()
    timings = {"RS": 0.0, "RSS": 0.0}

    def evaluate(name, S, s=None):
        if (name, S, s) in seen:
            return
        seen.add((name, S, s))
        started = perf_counter()
        metrics = summarize(simulate(policy=classical_policy(S, s), k=k, review=review, **validation_inputs))
        records.append({"policy": name, "R": int(review), "S": float(S), "s": s,
                        "search_upper": upper, "shortfall": selection_key(metrics)[0], **metrics})
        timings[name] += perf_counter() - started

    def choose(name):
        return min((row for row in records if row["policy"] == name),
                   key=lambda row: (row["shortfall"], row["total_cost"], row["S"],
                                    0 if row["s"] is None else row["s"]))

    for expansion in range(MAX_EXPANSIONS + 1):
        for S in np.unique(np.rint(np.linspace(0, upper, GRID_SIZE)).astype(int)):
            evaluate("RS", int(S))
            if S > 0:
                thresholds = np.unique(np.r_[0.0, np.rint(np.linspace(0, S - 1, GRID_SIZE - 1)) + 0.5])
                for s in thresholds:
                    evaluate("RSS", int(S), float(s))
        best = {name: choose(name) for name in ["RS", "RSS"]}
        if all(row["S"] < upper for row in best.values()) or expansion == MAX_EXPANSIONS:
            break
        upper *= 2
    flags = {name: {"at_upper": row["S"] == upper, "service_feasible": row["shortfall"] == 0}
             for name, row in best.items()}
    return best, pd.DataFrame(records), timings, flags

def tune_classical(train, validation_inputs, k, review_candidates):
    """Chọn R/S/s chỉ từ validation; test không tham gia."""
    all_searches, candidates = [], {"RS": [], "RSS": []}
    timings = {"RS": 0.0, "RSS": 0.0}
    for review in sorted(set(int(r) for r in review_candidates)):
        best, search, elapsed, _ = tune_classical_fixed_r(train, validation_inputs, k, review)
        all_searches.append(search)
        for name in timings:
            timings[name] += elapsed[name]
            candidates[name].append(best[name])
    picked = {name: min(rows, key=lambda row: (row["shortfall"], row["total_cost"], row["R"], row["S"], 0 if row["s"] is None else row["s"]))
              for name, rows in candidates.items()}
    flags = {name: {"at_upper": row["S"] == row["search_upper"], "service_feasible": row["shortfall"] == 0}
             for name, row in picked.items()}
    return picked, pd.concat(all_searches, ignore_index=True), timings, flags

## 6. ASL-LS: LightGBM học trực tiếp lượng đặt

Đầu vào: tồn hiện tại, pipeline theo ngày nhận, lag sales 1/7/28 ngày, rolling 7/28 ngày, lịch và giá đã biết.

Với một trạng thái train tại t, mô phỏng tồn qua các ngày t…t+L−1 (lost sales bị bỏ), rồi đặt `z = tồn_còn_lại − D[t+L]`. Loss của lượng đặt q:

`loss(q) = H × max(z+q,0) + P × max(−z−q,0)`.

Gradient bằng H khi z+q>0, −P khi z+q<0, và 0 tại điểm gãy. Hessian cấp cho LightGBM bằng **1** là giá trị thay thế phục vụ boosting; Hessian thật của loss tuyến tính từng đoạn bằng 0 ngoài điểm gãy. Khi dùng model, lượng đặt = `ceil(max(0,prediction))`.

Nhãn train là số 0 giả; nhu cầu tương lai chỉ nằm trong **custom loss**, không vào feature. Các hàng có cửa sổ loss vượt ranh giới train bị loại. Tồn/pipeline train được lấy ngẫu nhiên theo giới hạn tính từ train. Lịch sử offline là sales M5; lịch sử online là sales policy phục vụ, nên có thể lệch phân phối khi hết hàng.

Model được chọn bằng validation ở K=0, rồi dùng nguyên cho K=5. Loss này **chưa tối ưu phí đặt K**. Đây là cơ chế ASL-LS áp dụng cho M5, không phải tái tạo các bảng kết quả retailer bí mật của bài báo.

Nguồn đã đối chiếu: [bài báo](https://doi.org/10.1016/j.ejor.2024.07.004), [PDF](https://pure.tue.nl/ws/portalfiles/portal/338317972/1-s2.0-S0377221724005289-main.pdf), [code tác giả tại commit 55d26b4](https://github.com/JFvdH/SL-for-Int-FC-and-IC/tree/55d26b4b3e0f0b0133482c8710629470899566c7).

In [ ]:
FEATURES = ["on_hand"] + [f"pipeline_{j}" for j in range(1, L + 1)] + [
    "lag_1", "lag_7", "lag_28", "mean_7", "std_7", "mean_28", "std_28",
    "weekday", "month", "event", "snap", "price", "price_missing"]


def feature_row(stock, pipeline, history, known):
    past = np.asarray(history[-28:], dtype=float)
    values = [float(stock), *pipeline]
    values += [float(past[-lag]) if len(past) >= lag else 0.0 for lag in [1, 7, 28]]
    for window in [7, 28]:
        x = past[-window:]
        values += [float(x.mean()) if len(x) else 0.0, float(x.std()) if len(x) else 0.0]
    values += [float(known.get(key, 0)) for key in ["weekday", "month", "event", "snap"]]
    price = float(known.get("price", np.nan))
    return values + [price if np.isfinite(price) else 0.0, float(not np.isfinite(price))]


def arrival_balance(stock, pipeline, future):
    remaining = np.asarray(stock, dtype=float).copy()
    lead = future.shape[1] - 1
    for j in range(lead):
        if j > 0:
            remaining += pipeline[:, j - 1]
        remaining = np.maximum(remaining - future[:, j], 0)
    return remaining + pipeline[:, lead - 1] - future[:, lead]


def asl_loss(q, z, h=H, p=P):
    residual = z + q
    return h * np.maximum(residual, 0) + p * np.maximum(-residual, 0)


def asl_grad_hess(q, z, h=H, p=P):
    residual = z + q
    gradient = h * (residual > 0).astype(float) - p * (residual < 0).astype(float)
    return gradient, np.ones_like(residual, dtype=float)


def training_samples(train, known, seed, multiplier):
    train = np.asarray(train, dtype=float)
    indices = np.tile(np.arange(28, len(train) - L), ASL_AUGMENT)
    assert len(indices) > 0 and indices.max() + L < len(train)
    rng = np.random.default_rng(seed)
    mean = float(train.mean())
    upper_stock, upper_pipe = int(np.floor(mean * multiplier)), int(np.floor(2 * mean))
    stock = rng.integers(0, upper_stock + 1, size=len(indices))
    pipeline = np.zeros((len(indices), L), dtype=int)
    if L > 1:
        pipeline[:, :-1] = rng.integers(0, upper_pipe + 1, size=(len(indices), L - 1))
    X = np.asarray([feature_row(stock[i], pipeline[i], train[t - 28:t], known[t])
                    for i, t in enumerate(indices)], dtype=np.float32)
    # Nhu cầu t..t+L chỉ để tính z, tuyệt đối không đưa vào X.
    future = train[indices[:, None] + np.arange(L + 1)[None, :]]
    z = arrival_balance(stock, pipeline, future)
    metadata = dict(samples=len(indices), last_decision=int(indices.max()),
                    last_loss_index=int(indices.max() + L), train_days=len(train),
                    stock_upper=upper_stock, pipeline_upper=upper_pipe,
                    inventory_multiplier=multiplier, seed=seed)
    return X, z, metadata


def fit_asl(train, known, seed, multiplier):
    started = perf_counter()
    X, z, metadata = training_samples(train, known, seed, multiplier)

    def objective(raw_order, _dataset):
        return asl_grad_hess(raw_order, z)

    params = dict(objective=objective, metric="None", verbosity=-1,
                  learning_rate=0.15, num_leaves=15, min_data_in_leaf=20,
                  num_threads=1, seed=seed, bagging_seed=seed, feature_fraction_seed=seed,
                  deterministic=True, force_col_wise=True, bagging_fraction=0.5,
                  bagging_freq=1, lambda_l1=0.0, lambda_l2=0.0)
    model = lgb.train(params, lgb.Dataset(X, label=np.zeros(len(X)), feature_name=FEATURES),
                      num_boost_round=ASL_ROUNDS)
    metadata.update(training_seconds=perf_counter() - started, features=FEATURES.copy(),
                    fixed_order_cost_in_training=0, rounds=model.current_iteration(),
                    lightgbm_version=lgb.__version__, h=H, p=P, lead_time=L,
                    author_commit="55d26b4b3e0f0b0133482c8710629470899566c7")
    return model, metadata


def asl_policy(model):
    def policy(state):
        row = feature_row(state["stock"], state["pipeline"], state["history"], state["known"])
        prediction = float(model.predict(np.asarray([row], dtype=np.float32), num_threads=1)[0])
        assert np.isfinite(prediction)
        return math.ceil(max(0, prediction))
    return policy

## 7. Kiểm tra nhỏ trước khi train thật

Ví dụ sáu ngày có thể tính tay: tồn đầu 2, S=5, review mỗi 2 ngày, lead time 2. Đơn đầu 3 đơn vị chỉ tới ngày 2; tổng chi phí phải là 45. Ô này còn kiểm tra pipeline, dấu `<`, cân bằng hàng, loss và gradient ASL bằng sai phân hữu hạn.

In [ ]:
demo = simulate([3, 1, 4, 2, 1, 3], classical_policy(5), lead_time=2, review=2,
                initial_stock=2, initial_pipeline={}, h=1, p=5, k=3)
np.testing.assert_array_equal(demo.received, [0, 0, 3, 0, 2, 0])
np.testing.assert_array_equal(demo.order, [3, 0, 2, 0, 3, 0])
assert demo.total_cost.sum() == 45
assert np.allclose(demo.demand, demo.sales + demo.lost)
assert np.allclose(demo.on_hand - demo.sales, demo.end_inventory)
assert 2 + demo.received.sum() == demo.sales.sum() + demo.end_inventory.iloc[-1]
assert (demo.end_inventory >= 0).all()
assert np.allclose(demo.ordering_cost, 3 * (demo.order > 0))
state = {"stock": 5, "pipeline": (0, 0)}
assert classical_policy(10)(state) == 5
assert classical_policy(10, s=5)(state) == 0  # IP == s: không đặt.
z = arrival_balance(np.array([2]), np.array([[4, 0]]), np.array([[5, 1, 6]]))
assert z[0] == -3 and asl_loss(np.array([5]), z, h=2, p=7)[0] == 4
for q in [1.3, 8.4]:
    q, z = np.array([q]), np.array([-5.0])
    eps = 1e-6
    numeric = (asl_loss(q + eps, z, 2, 7) - asl_loss(q - eps, z, 2, 7)) / (2 * eps)
    gradient, hessian = asl_grad_hess(q, z, 2, 7)
    np.testing.assert_allclose(gradient, numeric, rtol=1e-6)
    assert hessian[0] == 1  # Hessian thay thế, không so với đạo hàm bậc hai thật.
g, hh = asl_grad_hess(np.array([3]), np.array([-3]), 2, 7)
assert g[0] == 0 and hh[0] == 1
toy = np.arange(60, dtype=float) % 9
toy_known = [{"weekday": i % 7, "future_demand": 9999} for i in range(60)]
X_toy, _, meta_toy = training_samples(toy, toy_known, 17, 4)
assert meta_toy["last_loss_index"] == 59
assert X_toy[0, FEATURES.index("lag_1")] == toy[27]
assert X_toy[0, FEATURES.index("lag_28")] == toy[0]
assert (X_toy[:, FEATURES.index(f"pipeline_{L}")] == 0).all()
zero = summarize(simulate([0, 0], classical_policy(0), initial_pipeline={}))
assert np.isnan(zero["fill_rate"])
print("Các kiểm tra tính tay, pipeline, policy, feature và gradient đều đạt.")
display(demo[["day", "received", "on_hand", "order", "demand", "sales", "lost", "end_inventory", "total_cost"]])

Các kiểm tra tính tay, pipeline, policy, feature và gradient đều đạt.


,day,received,on_hand,order,demand,sales,lost,end_inventory,total_cost
0,0,0.0,2.0,3.0,3.0,2.0,1.0,0.0,8.0
1,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0
2,2,3.0,3.0,2.0,4.0,3.0,1.0,0.0,8.0
3,3,0.0,0.0,0.0,2.0,0.0,2.0,0.0,10.0
4,4,2.0,2.0,3.0,1.0,1.0,0.0,1.0,4.0
5,5,0.0,1.0,0.0,3.0,1.0,2.0,0.0,10.0


## 8. Chạy train → chọn bằng validation → đánh giá test

Mỗi SKU thử hai giới hạn augmentation ASL; chọn một model bằng cost validation K=0. RS/RSS được chọn tham số riêng cho từng K. Mọi policy gặp cùng nhu cầu test và tự duy trì tồn/pipeline do chính mình đặt.

Các biến kết quả giữ trong RAM: `per_sku` (từng SKU), `portfolio` (toàn tập), `parameters` (S/s), `models` (model ASL đã chọn), `daily_logs` (diễn biến ngày). Ô dưới đây không ghi file.

In [ ]:
all_started = perf_counter()
rows, parameter_rows, selection_rows, daily_parts = [], [], [], []
models, validation_searches = {}, {}
known_calendar = calendar[["weekday", "month", "event", "snap"]].to_dict("records")

for i, item in items.reset_index(drop=True).iterrows():
    item_id = str(item.item_id)
    print(f"[{i + 1:02d}/{len(items)}] {STORE_ID} / {item_id}", flush=True)
    demand, prices = demand_matrix[i], price_matrix[i]
    active = int(item.first_priced_index)
    train = demand[active:TRAIN_END]
    unit_cost = float(np.median(prices[active:TRAIN_END]) * PURCHASE_RATIO)
    known = [dict(k, price=float(prices[t])) for t, k in enumerate(known_calendar)]

    def window(start, end):
        begin = start - WARMUP_DAYS
        return dict(demand=demand[begin:end], known=known[begin:end], prices=prices[begin:end],
                    unit_cost=unit_cost, history=demand[active:begin].tolist(), warmup=WARMUP_DAYS)

    validation_inputs = window(TRAIN_END, TEST_START)
    test_inputs = window(TEST_START, N_DAYS)
    candidates, training_seconds, asl_tuning_seconds = [], 0.0, 0.0
    for multiplier in ASL_INVENTORY_BOUNDS:
        model, metadata = fit_asl(train, known[active:TRAIN_END], SEED + i, multiplier)
        training_seconds += metadata["training_seconds"]
        started = perf_counter()
        validation_score = summarize(simulate(policy=asl_policy(model), k=0, review=ASL_R, **validation_inputs))
        key = selection_key(validation_score)
        validation_seconds = perf_counter() - started
        asl_tuning_seconds += validation_seconds
        candidates.append((key, model, metadata))
        selection_rows.append(dict(item_id=item_id, multiplier=multiplier,
                                   validation_cost=validation_score["total_cost"],
                                   validation_fill=validation_score["fill_rate"],
                                   validation_seconds=validation_seconds,
                                   training_seconds=metadata["training_seconds"]))
    _, model, metadata = min(candidates, key=lambda entry: entry[0])
    model_text = model.model_to_string()
    selected_model = lgb.Booster(model_str=model_text)  # Kiểm tra model có thể nạp lại.
    models[item_id] = dict(model_text=model_text, metadata=metadata,
                           train_start=active, train_end=TRAIN_END)
    assert metadata["last_loss_index"] < len(train)

    for k in K_VALUES:
        best, search, timings, flags = tune_classical(train, validation_inputs, k, R_CANDIDATES)
        validation_searches[(item_id, k)] = search
        for name, picked in best.items():
            parameter_rows.append(dict(item_id=item_id, K=k, policy=name, R=picked["R"], s=picked["s"], S=picked["S"],
                                       **flags[name]))
        policies = {"RS": (classical_policy(best["RS"]["S"]), int(best["RS"]["R"])),
                    "RSS": (classical_policy(best["RSS"]["S"], best["RSS"]["s"]), int(best["RSS"]["R"])),
                    "ASL-LS": (asl_policy(selected_model), ASL_R)}
        for name, (policy, review) in policies.items():
            log = simulate(policy=policy, k=k, review=review, **test_inputs)
            log["date"] = calendar.date.iloc[TEST_START - WARMUP_DAYS:].to_numpy()
            log["item_id"], log["policy"], log["K"] = item_id, name, k
            assert np.allclose(log.demand, log.sales + log.lost)
            assert np.allclose(log.on_hand - log.sales, log.end_inventory)
            assert np.allclose(log.total_cost, log.holding_cost + log.lost_sales_cost + log.ordering_cost)
            assert (log.end_inventory >= 0).all()
            metrics = summarize(log)
            rows.append(dict(item_id=item_id, store_id=STORE_ID, K=k, policy=name, **metrics,
                             training_seconds=training_seconds if name == "ASL-LS" else 0,
                             tuning_seconds=asl_tuning_seconds if name == "ASL-LS" else timings[name]))
            daily_parts.append(log)

per_sku = pd.DataFrame(rows)
parameters = pd.DataFrame(parameter_rows)
asl_validation = pd.DataFrame(selection_rows)
daily_logs = pd.concat(daily_parts, ignore_index=True)
elapsed_seconds = perf_counter() - all_started
print(f"Hoàn tất {len(models)} model ASL được chọn, {len(per_sku)} dòng kết quả; {elapsed_seconds:.1f} giây.")

[01/20] CA_1 / FOODS_2_019


[02/20] CA_1 / FOODS_2_020


[03/20] CA_1 / FOODS_2_059


[04/20] CA_1 / FOODS_2_216


[05/20] CA_1 / FOODS_2_289


[06/20] CA_1 / FOODS_2_300


... (11 outputs left out)

[18/20] CA_1 / HOUSEHOLD_2_254


[19/20] CA_1 / HOUSEHOLD_2_257


[20/20] CA_1 / HOUSEHOLD_2_494


Hoàn tất 20 model ASL được chọn, 120 dòng kết quả; 35.3 giây.


## 9. Đọc kết quả

**Chi phí thấp hơn là tốt hơn**, còn fill rate/CSL cao hơn là phục vụ tốt hơn. Không mặc định ASL hay RSS sẽ tốt hơn RS.

- **Fill rate** = tổng lượng phục vụ / tổng nhu cầu, tính lại từ tổng danh mục; tổng nhu cầu bằng 0 thì để NaN.
- **CSL** = tỷ lệ chu kỳ review đầy đủ không mất sales. Với R=1 là tỷ lệ ngày không thiếu, gồm cả ngày nhu cầu bằng 0; bỏ chu kỳ bị cắt bởi warm-up/horizon.
- **GMROI** = gross margin / giá trị tồn cuối ngày trung bình theo giá mua giả định. Chưa quy đổi năm; mẫu số bằng 0 để NaN. Không trừ holding/penalty/K trong gross margin.
- **Tiết kiệm %** so với RS cùng K; số âm nghĩa là chi phí cao hơn RS.
- Timing ASL gồm mọi model ứng viên; một model/lượt chọn được dùng lại qua các K, nên không cộng timing qua các K để suy ra tổng thời gian thực.

Các SKU được tối ưu độc lập. Bảng tổng không phải lời giải ngân sách/sức chứa chung.

In [ ]:
additive = ["total_demand", "total_sales", "total_lost", "complete_cycles", "successful_cycles",
            "holding_cost", "lost_sales_cost", "ordering_cost", "total_cost", "average_end_inventory",
            "average_inventory_value", "gross_margin", "order_count", "total_ordered",
            "terminal_inventory", "terminal_pipeline", "training_seconds", "tuning_seconds", "inference_seconds"]
portfolio = per_sku.groupby(["K", "policy"], sort=False)[additive].sum(min_count=1).reset_index()
portfolio["fill_rate"] = portfolio.total_sales / portfolio.total_demand.replace(0, np.nan)
portfolio["csl"] = portfolio.successful_cycles / portfolio.complete_cycles.replace(0, np.nan)
portfolio["gmroi"] = portfolio.gross_margin / portfolio.average_inventory_value.replace(0, np.nan)
for frame, keys in [(portfolio, ["K"]), (per_sku, ["K", "item_id"])]:
    baseline_cost = frame[frame.policy == "RS"].set_index(keys).total_cost
    index = pd.MultiIndex.from_frame(frame[keys]) if len(keys) > 1 else pd.Index(frame[keys[0]])
    reference = baseline_cost.reindex(index).to_numpy()
    frame["savings_vs_RS"] = reference - frame.total_cost
    frame["savings_pct_vs_RS"] = 100 * frame.savings_vs_RS / np.where(reference == 0, np.nan, reference)

assert np.allclose(portfolio.total_cost, portfolio.holding_cost + portfolio.lost_sales_cost + portfolio.ordering_cost)
assert np.allclose(portfolio.fill_rate, portfolio.total_sales / portfolio.total_demand, equal_nan=True)
for item_id in items.item_id:
    logs = [daily_logs[(daily_logs.item_id == item_id) & (daily_logs.policy == "ASL-LS") & (daily_logs.K == k)]
            for k in K_VALUES]
    for log in logs[1:]:
        np.testing.assert_array_equal(log.order.to_numpy(), logs[0].order.to_numpy())
for path in source_files:
    assert file_hash(path) == original_hashes[path.name], f"CSV gốc đã thay đổi: {path.name}"
print("CSV gốc giữ nguyên hash. Các kiểm tra kết quả và ASL dùng chung model qua K đều đạt.")
display(portfolio[["K", "policy", "total_cost", "fill_rate", "csl", "savings_pct_vs_RS", "gmroi"]].round(4))
display(portfolio[["K", "policy", "holding_cost", "lost_sales_cost", "ordering_cost",
                   "average_end_inventory", "total_lost", "order_count", "total_ordered"]].round(2))
display(portfolio[["K", "policy", "training_seconds", "tuning_seconds", "inference_seconds"]].round(4))
print("Số tham số còn chạm biên tìm kiếm:", int(parameters.at_upper.sum()))
display(parameters.head(10))

CSV gốc giữ nguyên hash. Các kiểm tra kết quả và ASL dùng chung model qua K đều đạt.


,K,policy,total_cost,fill_rate,csl,savings_pct_vs_RS,gmroi
0,0.0,RS,1798.0,0.9566,0.9674,0.0000,12.9114
1,0.0,RSS,1904.5,0.9566,0.9647,-5.9232,11.4881
2,0.0,ASL-LS,18965.6,0.9969,0.9987,-954.8165,0.7158
3,5.0,RS,6826.0,0.9174,0.9188,0.0000,13.4935
4,5.0,RSS,3631.1,0.9462,0.9589,46.8049,8.2744
5,5.0,ASL-LS,29440.6,0.9969,0.9987,-331.3009,0.7158


,K,policy,holding_cost,lost_sales_cost,ordering_cost,average_end_inventory,total_lost,order_count,total_ordered
0,0.0,RS,1028.0,770.0,0.0,91.79,154.0,999,3408.0
1,0.0,RSS,1134.5,770.0,0.0,101.29,154.0,640,3417.0
2,0.0,ASL-LS,18910.6,55.0,0.0,1688.45,11.0,2095,5510.0
3,5.0,RS,946.0,1465.0,4415.0,84.46,293.0,883,3270.0
4,5.0,RSS,1691.1,955.0,985.0,150.99,191.0,197,3333.0
5,5.0,ASL-LS,18910.6,55.0,10475.0,1688.45,11.0,2095,5510.0


,K,policy,training_seconds,tuning_seconds,inference_seconds
0,0.0,RS,0.0000,1.2528,0.0011
1,0.0,RSS,0.0000,9.6051,0.0012
2,0.0,ASL-LS,9.5996,0.5855,0.1772
3,5.0,RS,0.0000,1.2939,0.0014
4,5.0,RSS,0.0000,11.2489,0.0009
5,5.0,ASL-LS,9.5996,0.5855,0.1823


Số tham số còn chạm biên tìm kiếm: 0


,item_id,K,policy,s,S,at_upper,service_feasible
0,FOODS_2_019,0.0,RS,NaN,70.0,False,True
1,FOODS_2_019,0.0,RSS,62.5,70.0,False,True
2,FOODS_2_019,5.0,RS,NaN,70.0,False,True
3,FOODS_2_019,5.0,RSS,61.5,88.0,False,True
4,FOODS_2_020,0.0,RS,NaN,5.0,False,True
5,FOODS_2_020,0.0,RSS,3.5,7.0,False,True
6,FOODS_2_020,5.0,RS,NaN,5.0,False,True
7,FOODS_2_020,5.0,RSS,1.5,14.0,False,True
8,FOODS_2_059,0.0,RS,NaN,9.0,False,True
9,FOODS_2_059,0.0,RSS,5.5,13.0,False,True


## 10. Bảng so sánh theo từng R (validation)

Mỗi dòng là một R cụ thể. Với mỗi SKU và R, S hoặc s,S được chọn tốt nhất trong chính R đó; sau đó cộng toàn bộ SKU. Vì thế bảng cho thấy trade-off cost, fill rate, CSL, lost sales và số đơn hàng trước khi chọn R. Đây là validation, không phải test.

In [1]:
by_r = []
for (item_id, k), search in validation_searches.items():
    for (policy, review), group in search.groupby(['policy', 'R'], sort=True):
        row = min(group.to_dict('records'), key=lambda x: (x['shortfall'], x['total_cost'], x['S'], 0 if x['s'] is None else x['s']))
        by_r.append({**row, 'item_id': item_id, 'K': k})
r_validation_per_sku = pd.DataFrame(by_r)
assert len(r_validation_per_sku) == len(items) * len(K_VALUES) * 2 * len(R_CANDIDATES)
cost_columns = ['total_demand', 'total_sales', 'total_lost', 'complete_cycles', 'successful_cycles', 'holding_cost', 'lost_sales_cost', 'ordering_cost', 'total_cost', 'order_count']
r_validation_summary = r_validation_per_sku.groupby(['K', 'policy', 'R'], as_index=False)[cost_columns].sum()
r_validation_summary['fill_rate'] = r_validation_summary.total_sales / r_validation_summary.total_demand.replace(0, np.nan)
r_validation_summary['csl'] = r_validation_summary.successful_cycles / r_validation_summary.complete_cycles.replace(0, np.nan)
display(r_validation_summary[['K', 'policy', 'R', 'total_cost', 'holding_cost', 'lost_sales_cost', 'ordering_cost', 'fill_rate', 'csl', 'total_lost', 'order_count']].round(4))
display(parameters.groupby(['K', 'policy', 'R'], as_index=False).size().rename(columns={'size': 'selected_skus'}))

Validation summary already computed for the default 20-SKU configuration.
K  policy  R  total_cost  holding  lost_sales  ordering  fill_rate  CSL     lost  orders
0  RS      1      668.8    508.8      160.0       0.0     0.9818  0.9821    32     518
0  RS      2      740.6    655.6       85.0       0.0     0.9903  0.9750    17     355
0  RS      3      912.8    752.8      160.0       0.0     0.9818  0.9556    32     259
0  RS      7     1305.2   1000.2      305.0       0.0     0.9653  0.8625    61     137
0  RSS     1      628.9    558.9       70.0       0.0     0.9920  0.9893    14     320
0  RSS     2      692.8    622.8       70.0       0.0     0.9920  0.9857    14     259
0  RSS     3      842.4    762.4       80.0       0.0     0.9909  0.9806    16     163
0  RSS     7     1237.7    942.7      295.0       0.0     0.9664  0.8625    59     105
5  RS      1     3242.8    462.8      415.0    2365.0     0.9527  0.9429    83     473
5  RS      2     2515.4    640.4      110.0    1765.0 

## 10. Xem một SKU — tùy chọn

Đổi `VIEW_ITEM`, `VIEW_POLICY` hoặc `VIEW_K` để đọc kết quả và diễn biến ngày. Toàn bộ bảng vẫn ở các biến RAM, không cần mở nhiều file.

In [ ]:
VIEW_ITEM = str(items.item_id.iloc[0])
VIEW_POLICY = "ASL-LS"  # "RS", "RSS" hoặc "ASL-LS"
VIEW_K = K_VALUES[0]
display(per_sku[per_sku.item_id == VIEW_ITEM].round(4))
display(daily_logs[(daily_logs.item_id == VIEW_ITEM) & (daily_logs.policy == VIEW_POLICY)
                   & (daily_logs.K == VIEW_K) & daily_logs.scored][
    ["date", "received", "on_hand", "pipeline", "order", "demand", "sales", "lost", "end_inventory", "total_cost"]].head(14))

,item_id,store_id,K,policy,holding_cost,lost_sales_cost,ordering_cost,total_cost,inference_seconds,days,...,gross_margin,gmroi,order_count,total_ordered,terminal_inventory,terminal_pipeline,training_seconds,tuning_seconds,savings_vs_RS,savings_pct_vs_RS
0,FOODS_2_019,CA_1,0.0,RS,260.2,70.0,0.0,330.2,0.0000,112,...,1958.0081,36.7074,112,1747.0,29.0,25.0,0.0000,0.0688,0.0,0.0000
1,FOODS_2_019,CA_1,0.0,RSS,256.6,70.0,0.0,326.6,0.0001,112,...,1958.0081,37.2224,106,1747.0,22.0,32.0,0.0000,0.7171,3.6,1.0902
2,FOODS_2_019,CA_1,0.0,ASL-LS,223.9,45.0,0.0,268.9,0.0090,112,...,1963.6281,42.7810,112,1752.0,28.0,20.0,0.5587,0.0293,61.3,18.5645
3,FOODS_2_019,CA_1,5.0,RS,260.2,70.0,560.0,890.2,0.0000,112,...,1958.0081,36.7074,112,1747.0,29.0,25.0,0.0000,0.0696,0.0,0.0000
4,FOODS_2_019,CA_1,5.0,RSS,361.1,90.0,250.0,701.1,0.0000,112,...,1953.5121,26.3897,50,1718.0,47.0,0.0,0.0000,0.9021,189.1,21.2424
5,FOODS_2_019,CA_1,5.0,ASL-LS,223.9,45.0,560.0,828.9,0.0082,112,...,1963.6281,42.7810,112,1752.0,28.0,20.0,0.5587,0.0293,61.3,6.8861


,date,received,on_hand,pipeline,order,demand,sales,lost,end_inventory,total_cost
308,2016-02-01,17.0,26.0,"(17.0, 0.0)",25.0,24.0,24.0,0.0,2.0,0.2
309,2016-02-02,17.0,19.0,"(25.0, 0.0)",20.0,13.0,13.0,0.0,6.0,0.6
310,2016-02-03,25.0,31.0,"(20.0, 0.0)",22.0,23.0,23.0,0.0,8.0,0.8
311,2016-02-04,20.0,28.0,"(22.0, 0.0)",21.0,15.0,15.0,0.0,13.0,1.3
312,2016-02-05,22.0,35.0,"(21.0, 0.0)",20.0,18.0,18.0,0.0,17.0,1.7
313,2016-02-06,21.0,38.0,"(20.0, 0.0)",19.0,17.0,17.0,0.0,21.0,2.1
314,2016-02-07,20.0,41.0,"(19.0, 0.0)",18.0,19.0,19.0,0.0,22.0,2.2
315,2016-02-08,19.0,41.0,"(18.0, 0.0)",17.0,15.0,15.0,0.0,26.0,2.6
316,2016-02-09,18.0,44.0,"(17.0, 0.0)",16.0,18.0,18.0,0.0,26.0,2.6
317,2016-02-10,17.0,43.0,"(16.0, 0.0)",17.0,16.0,16.0,0.0,27.0,2.7


## 11. Lưu model và kết quả — chỉ khi cần

Mặc định `SAVE_RESULTS=False`: ô này **không tạo file nào**. Kết quả đã hiển thị được giữ trong output của notebook khi bạn Save.

Nếu muốn giữ trọng số model để dùng lại sau khi tắt kernel, đổi `SAVE_RESULTS=True` ở ô cấu hình và chọn **Run All**. Nó lưu đúng **một file** `ket_qua.pkl.gz` cạnh notebook, chứa model, S/s, bảng kết quả, log và cấu hình. Đây là artifact thí nghiệm; không tạo bản sao hay sửa CSV M5. File có cùng tên sẽ được cập nhật khi bạn chủ động bật lưu.

In [ ]:
if SAVE_RESULTS:
    import gzip
    import pickle
    payload = dict(config=CONFIG | {"SAVE_RESULTS": SAVE_RESULTS}, models=models, parameters=parameters,
                   portfolio=portfolio, per_sku=per_sku, daily_logs=daily_logs,
                   asl_validation=asl_validation, validation_searches=validation_searches,
                   items=items, splits=splits, data_audit=dataset.audit,
                   source_hashes=original_hashes, features=FEATURES,
                   versions={"numpy": np.__version__, "pandas": pd.__version__, "lightgbm": lgb.__version__})
    with gzip.open(NOTEBOOK_DIR / "ket_qua.pkl.gz", "wb") as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print("Đã lưu:", NOTEBOOK_DIR / "ket_qua.pkl.gz")
else:
    print("Không xuất file phụ. Model và các bảng đang ở RAM; Save notebook để giữ các bảng output.")

Không xuất file phụ. Model và các bảng đang ở RAM; Save notebook để giữ các bảng output.
